# CTO from scratch sur ISIC 2018

Ce notebook entraîne un U-Net de référence et CTO sur ISIC 2018, les compare (tableau et graphes),
puis teste leur généralisation sur PH2 sans réentraînement.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
EPOCHS = 90

DRIVE = '/content/drive/MyDrive'
RUNS = f'{DRIVE}/cto-runs/{EPOCHS}ep'   # checkpoints et résultats de cette série d'entraînements
DATA_DRIVE = f'{DRIVE}/isic2018_256'    # ISIC 2018 préparé (environ 700 Mo)
PH2_DRIVE = f'{DRIVE}/ph2_256'          # PH2 préparé
DATA, PH2 = '/content/isic2018_256', '/content/ph2_256'   

In [ ]:
# Code du projet
!rm -rf /content/cto-scratch && unzip -q -o "{DRIVE}/cto-scratch.zip" -d /content
%cd /content/cto-scratch
!pip install -q timm albumentations thop

In [ ]:
import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun (active le GPU)')
!python -m pytest tests -q

## 1. Données ISIC 2018

À faire une seule fois. La cellule télécharge ISIC 2018 (tâche 1, 2594 images, environ 11 Go), réduit tout en 256x256
et enregistre le résultat sur Drive.

In [ ]:
if not os.path.exists(f'{DATA_DRIVE}/img.npy'):
    BASE = 'https://isic-archive.s3.amazonaws.com/challenges/2018'
    !wget -q --show-progress -P /content/raw {BASE}/ISIC2018_Task1-2_Training_Input.zip
    !wget -q --show-progress -P /content/raw {BASE}/ISIC2018_Task1_Training_GroundTruth.zip
    !unzip -q -o /content/raw/ISIC2018_Task1-2_Training_Input.zip -d /content/raw
    !unzip -q -o /content/raw/ISIC2018_Task1_Training_GroundTruth.zip -d /content/raw
    !python data/prepare_isic.py --images /content/raw/ISIC2018_Task1-2_Training_Input --masks /content/raw/ISIC2018_Task1_Training_GroundTruth --out "{DATA_DRIVE}"
!mkdir -p {DATA} && cp -n "{DATA_DRIVE}"/*.npy {DATA}/
!ls -lh {DATA}

## 2. Entraînement du U-Net de référence

Le Dice affiché à chaque époque est celui de validation (415 images). Le jeu de test (518 images) n'est utilisé qu'à la section 4.

In [ ]:
!python train.py --name unet --model unet --epochs {EPOCHS} --data-dir {DATA} --save-dir "{RUNS}" --fold 0

## 3. Entraînement de CTO

Transformer sur x4, BIM avec attention sur l'incertitude, loss de contours Dice.

In [ ]:
!python train.py --name cto --epochs {EPOCHS} --data-dir {DATA} --save-dir "{RUNS}" --fold 0

## 4. Test sur ISIC 2018 et comparaison

Évalue chaque modèle entraîné sur les 518 images de test, puis produit dans le dossier `comparison` :
le tableau comparatif, le Dice et l'IoU de chaque modèle, les courbes d'entraînement, le Dice selon la taille du modèle
et la figure qualitative (image, vérité terrain, prédictions).

In [ ]:
for run in sorted(glob.glob(f'{RUNS}/*/fold_0')):
    if os.path.exists(f'{run}/best.pth'):
        !python test.py --run-dir "{run}" --data-dir {DATA} --save-pred
!python compare.py --runs-dir "{RUNS}" --data-dir {DATA}

In [ ]:
from IPython.display import Image, Markdown, display
OUT = f'{RUNS}/comparison'
display(Markdown(open(f'{OUT}/results.md').read()))
for name in ['scores', 'training_curves', 'dice_vs_params', 'qualitative']:
    if os.path.exists(f'{OUT}/{name}.png'):
        display(Image(f'{OUT}/{name}.png', width=900))

## 5. Généralisation sur PH2, sans réentraînement

PH2 contient 200 images de dermoscopie d'un autre hôpital. Les modèles entraînés sur ISIC 2018 y sont évalués tels quels.

In [ ]:
if not os.path.exists(f'{PH2_DRIVE}/img.npy'):
    RAR = f'{DRIVE}/PH2Dataset.rar'
    if not os.path.exists(RAR):
        RAR = '/content/PH2Dataset.rar'
        !wget -q --show-progress -O {RAR} "https://www.dropbox.com/s/k88qukc20ljnbuo/PH2Dataset.rar?dl=1"
    !apt-get -qq install -y unrar > /dev/null
    !mkdir -p /content/ph2_raw && unrar x -o+ -inul "{RAR}" /content/ph2_raw/
    !python data/prepare_ph2.py --root /content/ph2_raw --out "{PH2_DRIVE}"
!mkdir -p {PH2} && cp -n "{PH2_DRIVE}"/*.npy {PH2}/
!ls -lh {PH2}

In [ ]:
# Évalue chaque modèle sur les 200 images de PH2, puis compare avec ISIC 2018
for run in sorted(glob.glob(f'{RUNS}/*/fold_0')):
    if os.path.exists(f'{run}/best.pth'):
        !python test.py --run-dir "{run}" --data-dir {DATA} --external-dir {PH2} --external-name ph2 --save-pred
!python compare.py --runs-dir "{RUNS}" --data-dir {DATA} --external-dir {PH2} --external-name ph2

In [ ]:
display(Markdown(open(f'{OUT}/generalization_ph2.md').read()))
for name in ['generalization_ph2', 'qualitative_ph2']:
    if os.path.exists(f'{OUT}/{name}.png'):
        display(Image(f'{OUT}/{name}.png', width=900))